In [5]:
# ============================================================
# 🦾 MASTER DASHBOARD: Research Edition (Dynamics + Kinematics)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D 
import ipywidgets as widgets
from IPython.display import display, clear_output
import glob
import os
import re

# 1. Setup & Geometry (Updated for Rev 5.2)
BASE_LOG_DIR = os.path.join("..", "..", "logs")

# Hardware Geometry
L_BASE = 10.0   # cm (Updated)
L1 = 20.0       # cm (Updated)
L2 = 20.0       # cm (Updated)
MAX_ALLOWED_ERROR = 0.5 # cm

# 2. Define the Main Analysis Logic
def load_and_analyze(run_folder_path):
    clear_output(wait=True)
    
    run_name = os.path.basename(run_folder_path)
    is_3d = False
    has_dynamics = False
    
    # --- DETECT LOG TYPE ---
    # Priority 1: New Dynamics Log
    log_dyn = glob.glob(os.path.join(run_folder_path, "arm_dynamics_log.csv"))
    # Priority 2: Standard 3D Log
    log_3d = glob.glob(os.path.join(run_folder_path, "arm_3d_log.csv"))
    # Priority 3: Legacy 2D Log
    log_2d = glob.glob(os.path.join(run_folder_path, "arm_2link_log.csv"))
    
    if log_dyn:
        df = pd.read_csv(log_dyn[0])
        is_3d = True
        has_dynamics = True
        print(f"📂 Loaded RESEARCH DATA (Dynamics): {run_name}")
    elif log_3d:
        df = pd.read_csv(log_3d[0])
        is_3d = True
        print(f"📂 Loaded Standard 3D Data: {run_name}")
    elif log_2d:
        df = pd.read_csv(log_2d[0])
        print(f"📂 Loaded 2D Data: {run_name}")
    else:
        print(f"❌ No data found in {run_name}")
        return

    # Normalize Time
    if 'timestamp_ms' in df.columns:
        df['time_sec'] = df['timestamp_ms'] / 1000.0
    else:
        df['time_sec'] = np.arange(len(df)) * 0.002 # Fallback
    
    # --- PRE-CALCULATE GEOMETRY (JOINT POSITIONS) ---
    try:
        if is_3d:
            rad_base = np.radians(df['base_deg'])
            rad_shld = np.radians(df['shoulder_deg'])
            
            # Elbow Position Calculation
            r_elbw = L1 * np.cos(rad_shld)
            df['elbow_z'] = L_BASE + L1 * np.sin(rad_shld)
            df['elbow_x'] = r_elbw * np.cos(rad_base)
            df['elbow_y'] = r_elbw * np.sin(rad_base)
        else:
            # Fallback for 2D logs
            rad_m1 = np.radians(df['motor1_pos']) if 'motor1_pos' in df else np.zeros(len(df))
            df['elbow_x'] = L1 * np.cos(rad_m1)
            df['elbow_y'] = L1 * np.sin(rad_m1)
            df['elbow_z'] = 0 
            df['base_deg'] = 0 
            df['shoulder_deg'] = 0
            
    except Exception as e:
        print(f"⚠️ Warning: Geometry calculation issue: {e}")

    # --- METRICS ---
    duration = df['time_sec'].max()
    max_error = df['error'].max() if 'error' in df else 0.0
    avg_error = df['error'].mean() if 'error' in df else 0.0
    verdict = "PASS" if max_error <= MAX_ALLOWED_ERROR else "FAIL"
    
    # Dynamics Metrics
    max_torque_s = df['torque_shoulder'].max() if has_dynamics else 0
    max_torque_e = df['torque_elbow'].max() if has_dynamics else 0
    

    # ========================================================
    # 📊 PART 1: STATIC ANALYSIS REPORT (v5.3)
    # ========================================================
    try:
        fig_static = plt.figure(figsize=(14, 10))
        gs = fig_static.add_gridspec(2, 2)
        
        # --- Plot 1: 3D Path (Top Left) ---
        if is_3d:
            ax1 = fig_static.add_subplot(gs[0, 0], projection='3d')
            ax1.set_title(f"1. 3D Spatial Path", fontweight='bold')
            ax1.plot(df['target_x'], df['target_y'], df['target_z'], 'g--', label='Target', alpha=0.5)
            ax1.plot(df['real_x'], df['real_y'], df['real_z'], 'b-', alpha=0.8, label='Actual', lw=2)
            ax1.set_zlabel("Z (cm)")
            # Skeleton (Final Frame)
            last = df.iloc[-1]
            ax1.plot([0, 0, last['elbow_x'], last['real_x']], 
                     [0, 0, last['elbow_y'], last['real_y']], 
                     [0, L_BASE, last['elbow_z'], last['real_z']], 'r-o', lw=3)
        else:
            ax1 = fig_static.add_subplot(gs[0, 0])
            ax1.plot(df['target_x'], df['target_y'], 'g--')
            ax1.plot(df['real_x'], df['real_y'], 'b-')

        ax1.legend()
        ax1.set_xlabel("X (cm)")
        ax1.set_ylabel("Y (cm)")

        # --- Plot 2: SAFETY FACTOR (Top Right) ---
        if has_dynamics:
            # Calculate Min SF for Title
            sf_shoulder = df['limit_shoulder'] / (df['torque_shoulder'].abs() + 0.01)
            sf_elbow = df['limit_elbow'] / (df['torque_elbow'].abs() + 0.01)
            min_sf = min(sf_shoulder.min(), sf_elbow.min())
            
            ax2 = fig_static.add_subplot(gs[0, 1])
            ax2.set_title(f"2. Safety Factor (Min: {min_sf:.1f}x)", fontweight='bold', color='darkblue')
            
            ax2.plot(df['time_sec'], sf_shoulder, 'r-', label='Shoulder SF')
            ax2.plot(df['time_sec'], sf_elbow, 'b-', label='Elbow SF')
            
            # Danger Zone Line
            ax2.axhline(1.0, color='k', linestyle='--', linewidth=2, label='FAILURE (1.0)')
            ax2.fill_between(df['time_sec'], 0, 1.0, color='red', alpha=0.2)
            
            ax2.set_ylim(0, 10) 
            ax2.set_ylabel("Safety Factor (Ratio)")
            ax2.set_xlabel("Time (s)") # <--- ADDED LABEL
            ax2.legend(loc='upper right')
            ax2.grid(True, alpha=0.3)
        else:
            ax2 = fig_static.add_subplot(gs[0, 1])
            ax2.text(0.3, 0.5, "No Dynamics Data")

        # --- Plot 3: Error (Bottom Left) ---
        ax3 = fig_static.add_subplot(gs[1, 0])
        ax3.set_title("3. Tracking Lag (Error)", fontweight='bold')
        ax3.plot(df['time_sec'], df['error'], 'k-', lw=1)
        ax3.axhline(MAX_ALLOWED_ERROR, color='r', linestyle=':', label='Max Spec')
        ax3.set_ylabel("Error (cm)")
        ax3.set_xlabel("Time (s)")
        ax3.grid(True, alpha=0.3)
        
        # --- Plot 4: Summary Text (Bottom Right) ---
        ax_text = fig_static.add_subplot(gs[1, 1])
        ax_text.axis('off')
        
        # Calculate RMS Error (Root Mean Square) - Standard Engineering Metric
        rms_error = np.sqrt((df['error']**2).mean())

        dyn_text = ""
        if has_dynamics:
            min_sf_s = sf_shoulder.min()
            min_sf_e = sf_elbow.min()
            dyn_text = f"""
        PHYSICS PERFORMANCE:
        --------------------
        Shoulder Min SF: {min_sf_s:.2f}x
        Elbow Min SF:    {min_sf_e:.2f}x
        Gravity Load:    {df['torque_shoulder'].abs().max():.2f} Nm (Max)
            """

        summary_text = f"""
        RUN METRICS: {run_name}
        -----------------------------
        Duration:   {duration:.2f} s
        Samples:    {len(df)}
        
        ACCURACY:
        ----------------
        Max Error:  {max_error:.4f} cm
        Avg Error:  {avg_error:.4f} cm
        RMS Error:  {rms_error:.4f} cm
        Result:     {verdict}
        {dyn_text}
        """
        
        ax_text.text(0.05, 0.5, summary_text, fontsize=11, family='monospace', va='center')

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plot Error: {e}")
        import traceback
        traceback.print_exc()

    # ========================================================
    # 🎮 PART 2: 3D INTERACTIVE REPLAY
    # ========================================================
    print("\n👇 3D DIGITAL TWIN (Drag Slider to Replay) 👇")
    
    def plot_3d_frame(frame_index, elev, azim):
        try:
            fig_3d = plt.figure(figsize=(10, 8))
            ax = fig_3d.add_subplot(111, projection='3d')
            
            if frame_index >= len(df): frame_index = len(df) - 1
            row = df.iloc[frame_index]
            
            # Determine Color based on STRESS (Torque)
            shldr_color = 'orange'
            elbw_color = 'orange'
            
            if has_dynamics:
                # If torque is > 90% of limit, turn RED
                if abs(row['torque_shoulder']) > (row['limit_shoulder'] * 0.9): shldr_color = 'red'
                if abs(row['torque_elbow']) > (row['limit_elbow'] * 0.9): elbw_color = 'red'

            # Set view limits
            ax.set_xlim(-25, 35)
            ax.set_ylim(-25, 35)
            ax.set_zlim(0, 40)
            
            info = f"t={row['time_sec']:.2f}s"
            if has_dynamics:
                info += f" | Trq: {row['torque_shoulder']:.1f}Nm"
            ax.set_title(info)
            
            # Floor
            ax.plot([-25, 35, 35, -25, -25], [-25, -25, 35, 35, -25], [0,0,0,0,0], 'k-', alpha=0.1) 
            
            # Trace
            hist = df.iloc[:frame_index+1:10] # Downsample trace
            ax.plot(df['target_x'], df['target_y'], df['target_z'], 'g--', alpha=0.1)
            ax.plot(hist['real_x'], hist['real_y'], hist['real_z'], 'b-', alpha=0.4)

            # Robot Skeleton
            if is_3d:
                # Base -> Shoulder
                ax.plot([0, 0], [0, 0], [0, L_BASE], 'k-', lw=4)
                
                # Shoulder -> Elbow
                ax.plot([0, row['elbow_x']], [0, row['elbow_y']], [L_BASE, row['elbow_z']], 
                        color=shldr_color, lw=5, solid_capstyle='round')
                
                # Elbow -> Wrist
                ax.plot([row['elbow_x'], row['real_x']], [row['elbow_y'], row['real_y']], 
                        [row['elbow_z'], row['real_z']], 
                        color=elbw_color, lw=4, solid_capstyle='round')
                
                # Joints
                ax.scatter([0, row['elbow_x'], row['real_x']], 
                           [0, row['elbow_y'], row['real_y']], 
                           [L_BASE, row['elbow_z'], row['real_z']], 
                           color='k', s=50)
            
            ax.view_init(elev=elev, azim=azim)
            plt.show()
        except Exception as e:
            print(f"Frame Error: {e}")

    # --- WIDGETS ---
    play = widgets.Play(
        value=0, min=0, max=len(df)-1,
        step=10, interval=20, description="Play"
    )
    slider_time = widgets.IntSlider(min=0, max=len(df)-1, step=10, value=0, layout=widgets.Layout(width='60%'))
    widgets.jslink((play, 'value'), (slider_time, 'value'))
    
    slider_elev = widgets.IntSlider(min=0, max=90, step=5, value=25, description='Elev')
    slider_azim = widgets.IntSlider(min=0, max=360, step=5, value=300, description='Azim')
    
    ui = widgets.VBox([
        widgets.HBox([play, slider_time]), 
        widgets.HBox([slider_elev, slider_azim])
    ])
    
    out = widgets.interactive_output(plot_3d_frame, {
        'frame_index': slider_time, 
        'elev': slider_elev, 
        'azim': slider_azim
    })
    
    display(ui, out)

# 3. Main Selection Logic
all_folders = [f.path for f in os.scandir(BASE_LOG_DIR) if f.is_dir()]
arm_folders = sorted(
    [f for f in all_folders if "arm_run" in os.path.basename(f)],
    key=lambda x: int(re.search(r"arm_run_?(\d+)", os.path.basename(x)).group(1)) 
                  if re.search(r"arm_run_?(\d+)", os.path.basename(x)) else 0
)

dashboard_output = widgets.Output()

if not arm_folders:
    print("❌ No 'arm_run' folders found.")
else:
    # Default to the most recent run
    dropdown = widgets.Dropdown(
        options=[(os.path.basename(f), f) for f in arm_folders],
        value=arm_folders[-1],
        description='Select Run:',
        layout=widgets.Layout(width='50%')
    )
    
    def on_run_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with dashboard_output:
                load_and_analyze(change['new'])

    dropdown.observe(on_run_change)
    display(dropdown, dashboard_output)
    
    # Init Load
    with dashboard_output:
        load_and_analyze(arm_folders[-1])

Dropdown(description='Select Run:', index=1, layout=Layout(width='50%'), options=(('arm_run_3d_8', '..\\..\\lo…

Output()